# 07 — Train & Evaluate

Trains A-E x {linear, mlp2} across repeated spatial k-fold (5 folds x 3
repeats), plus B-E's ablation variant (9 scenario-variants total),
checkpointed per fold so a disconnect resumes rather than restarts.
Then: aggregate results (all scenarios reported, per your standing rule),
head-depth comparison, and the 15 non-F pre-specified pairs through
Wilcoxon (primary) + Nadeau-Bengio (secondary), Holm-Bonferroni corrected,
separately for PR-AUC and AUROC.

**Scenario F and its 5 F-related pairs are skipped — deferred until F
exists.** Scenario G (XGBoost) runs as its own separate path.

GPU recommended, not required (graphs are small).

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/eval.yaml") as f:
    eval_cfg = yaml.safe_load(f)

PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints"
METRICS_DIR = OUTPUTS_DIR / "metrics"
SIG_DIR = OUTPUTS_DIR / "significance"
for d in [CHECKPOINT_DIR, METRICS_DIR, SIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
config = {"batch_size": eval_cfg.get("batch_size") or 32,
          "epoch_cap": eval_cfg.get("epoch_cap") or 150,
          "patience": eval_cfg.get("patience") or 20}
print(f"Device: {device} | config: {config}")

In [ ]:
import pandas as pd
import graph_datasets as ds
import train as tr
import evaluate as ev
import models

index_df = pd.read_parquet(PROCESSED_DIR / "dataset_index.parquet")
fold_cols = [c for c in index_df.columns if c.startswith("fold_rep")]
dataset = ds.DualGraphDataset(index_df, PROCESSED_DIR / "svg_graphs", PROCESSED_DIR / "tvg_graphs")
print(f"Dataset: {len(dataset)} points, folds: {fold_cols}")

svg_kwargs = dict(hidden_dim=64, heads=4, num_layers=2, dropout=0.35,
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2, cat_embed_dim=2)
tvg_kwargs = dict(hidden_dim=64, heads=4, num_layers=2, dropout=0.35,
                   building_type_vocab=58, highway_vocab=13,
                   building_type_embed_dim=8, highway_embed_dim=4)

In [ ]:
# ── Train every primary scenario x head depth ─────────────────────────
PRIMARY_SCENARIOS = ["A", "B", "C", "D", "E"]
HEAD_DEPTHS = ["linear", "mlp2"]

all_results = {}
for scenario in PRIMARY_SCENARIOS:
    for depth in HEAD_DEPTHS:
        key = f"{scenario}_{depth}"
        print(f"\n=== {key} ===")
        results = tr.run_scenario(scenario, depth, use_ablation=False, dataset=dataset,
                                   fold_cols=fold_cols, config=config, svg_kwargs=svg_kwargs,
                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
        all_results[key] = results
        print(f"  {len(results)} fold-runs complete.")

In [ ]:
# ── Ablation: B-E only (F deferred) ────────────────────────────────────
for scenario in ["B", "C", "D", "E"]:
    for depth in HEAD_DEPTHS:
        key = f"{scenario}_{depth}_ablation"
        print(f"\n=== {key} ===")
        results = tr.run_scenario(scenario, depth, use_ablation=True, dataset=dataset,
                                   fold_cols=fold_cols, config=config, svg_kwargs=svg_kwargs,
                                   tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
        all_results[key] = results
        print(f"  {len(results)} fold-runs complete.")

In [ ]:
# ── Scenario G: XGBoost, separate path ───────────────────────────────
import baseline_features
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

feat_table = baseline_features.build_feature_table(index_df["point_id"].tolist(),
                                                     PROCESSED_DIR / "svg_graphs", PROCESSED_DIR / "tvg_graphs", torch)
feat_table = feat_table.merge(index_df[["point_id", "label"] + fold_cols], on="point_id")

g_results = []
for fold_col in fold_cols:
    for fold_id in sorted(feat_table[fold_col].unique()):
        test = feat_table[feat_table[fold_col] == fold_id]
        train_val = feat_table[feat_table[fold_col] != fold_id]
        train_, val_ = train_test_split(train_val, test_size=0.15, random_state=42)

        feature_cols = [c for c in feat_table.columns if c not in ["point_id", "label"] + fold_cols]
        clf = XGBClassifier(n_estimators=200, max_depth=4, eval_metric="aucpr", random_state=42)
        clf.fit(train_[feature_cols], train_["label"])
        prob = clf.predict_proba(test[feature_cols])[:, 1]
        g_results.append({"fold_col": fold_col, "fold_id": int(fold_id),
                           **ev.compute_metrics(test["label"].values, prob)})

all_results["G"] = g_results
print(f"Scenario G: {len(g_results)} fold-runs complete.")

In [ ]:
# ── Aggregate + report EVERY scenario, regardless of performance ────────
agg_rows = []
for key, results in all_results.items():
    agg = ev.aggregate_fold_results(results)
    row = {"scenario": key}
    for metric, (mean, std) in agg.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(METRICS_DIR / "all_scenarios_summary.csv", index=False)
display(agg_df)

In [ ]:
# ── Head-depth comparison: pick a winner per scenario using mean PR-AUC,
#    then use ONE consistent depth for the formal pairwise comparisons ──
depth_compare = []
for scenario in PRIMARY_SCENARIOS:
    lin = agg_df[agg_df["scenario"] == f"{scenario}_linear"]["pr_auc_mean"].iloc[0]
    mlp = agg_df[agg_df["scenario"] == f"{scenario}_mlp2"]["pr_auc_mean"].iloc[0]
    depth_compare.append({"scenario": scenario, "linear_pr_auc": lin, "mlp2_pr_auc": mlp})

depth_df = pd.DataFrame(depth_compare)
display(depth_df)

WINNING_DEPTH = "linear" if depth_df["linear_pr_auc"].mean() >= depth_df["mlp2_pr_auc"].mean() else "mlp2"
print(f"\nWinning head depth (by mean PR-AUC across scenarios): {WINNING_DEPTH}")
print("Used for the formal 15-pair comparison below — same depth across every scenario, per the fairness rule.")

In [ ]:
# ── Formal comparison: the 15 non-F pre-specified pairs, PR-AUC and
#    AUROC as CO-EQUAL, each with its own Holm-Bonferroni correction ────
ALL_PAIRS = eval_cfg["pairs"]
NON_F_PAIRS = [p for p in ALL_PAIRS if "F" not in p]
print(f"{len(NON_F_PAIRS)} / {len(ALL_PAIRS)} pairs testable now (F-involving pairs deferred)")

def _key(scenario_or_plus):
    if scenario_or_plus.endswith("+"):
        return f"{scenario_or_plus[:-1]}_{WINNING_DEPTH}_ablation"
    return f"{scenario_or_plus}_{WINNING_DEPTH}"

n_train_approx = int(len(dataset) * 0.85 * (1 - 1 / len(index_df[fold_cols[0]].unique())))
n_test_approx = int(len(dataset) / len(index_df[fold_cols[0]].unique()))

for metric in ["pr_auc", "auroc"]:
    print(f"\n{'='*20} {metric.upper()} {'='*20}")
    fold_scores = {}
    for scenario in PRIMARY_SCENARIOS + ["G"]:
        key = scenario if scenario == "G" else _key(scenario)
        if key in all_results:
            fold_scores[scenario] = [r[metric] for r in all_results[key]]
    for scenario in ["B", "C", "D", "E"]:
        key = f"{scenario}_{WINNING_DEPTH}_ablation"
        if key in all_results:
            fold_scores[f"{scenario}+"] = [r[metric] for r in all_results[key]]

    comparison_df = ev.run_comparison(fold_scores, NON_F_PAIRS, n_train_approx, n_test_approx)
    comparison_df.to_csv(SIG_DIR / f"{metric}_comparison.csv", index=False)
    display(comparison_df)

In [ ]:
print("Deferred to when F exists: C-F, D-F, E-F, G-F, F-F+ (5 pairs), and F's ablation.")
print()
print("Next: 08_interpretability.ipynb")